# Communication-Efficient Distributed Vector Memory
## Investigating Adaptive Routing Across Partitioned Semantic Memories

**Working Title:** Communication-Efficient Distributed Vector Memory: Investigating Adaptive Routing Across Partitioned Semantic Memories

**Core Research Question:** Can a distributed semantic memory with adaptive inter-partition communication achieve retrieval performance comparable to a centralized vector database while requiring significantly less communication?

**Paper Reference:** TurboQuant (ICLR 2026) — data-oblivious quantizer with near-optimal distortion, no training phase.

**This notebook implements:**
1. Embedding generation for a 100K-document corpus from DBpedia 14 (sampled from 560K)
2. Semantic space partitioning into independent memories
3. Local TurboVec index construction per partition
4. Three retrieval strategies: Global (baseline), Local (baseline), Adaptive Communication (proposed)
5. Four adaptive routing policies: Confidence Threshold, Top-N Neighbors, Progressive Expansion, Budgeted Communication
6. Evaluation: retrieval quality (Recall@10, MRR, NDCG@10) vs communication cost

**Dataset:** DBpedia 14 — 100K documents sampled from 560K across 14 semantic classes (Company, EducationalInstitution, Artist, Athlete, OfficeHolder, MeanOfTransportation, Building, NaturalPlace, Village, Animal, Plant, Album, Film, WrittenWork)

## 1. Setup and Installation

In [ ]:
!pip install -q turbovec sentence-transformers scikit-learn matplotlib seaborn datasets

In [ ]:
import numpy as np
import time
import warnings
import glob
import json
import os
from collections import defaultdict

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import rankdata

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('All imports successful.')

In [ ]:
# Constants
N_DOCS = 100_000  # Use 100K for manageable runtime on Kaggle CPU (train split is 560K, total is 630K)
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
EMBEDDING_DIM = 384
BIT_WIDTH = 4
TOP_K = 10
N_QUERIES = 100
PARTITION_COUNTS = [2, 4, 8, 16]

print(f'Dataset: DBpedia 14 ({N_DOCS:,} documents)')
print(f'Embedding model: {EMBEDDING_MODEL} (dim={EMBEDDING_DIM})')
print(f'TurboVec bit width: {BIT_WIDTH}-bit ({2**BIT_WIDTH} quantization levels)')
print(f'Top-K: {TOP_K}')
print(f'Query count: {N_QUERIES}')
print(f'Partition counts: {PARTITION_COUNTS}')

## 2. Dataset Loading and Preparation

DBpedia 14: 560K training + 70K test documents across 14 semantic classes.
We load from HuggingFace and combine train+test for a unified corpus.

In [ ]:
from datasets import load_dataset

print('Loading DBpedia 14 from HuggingFace...')
t0 = time.time()
ds = load_dataset('fancyzhx/dbpedia_14')
load_time = time.time() - t0
print(f'Dataset loaded in {load_time:.1f}s')

# Combine train and test splits (convert to lists for concatenation)
train_texts = list(ds['train']['content'])
train_labels = list(ds['train']['label'])
test_texts = list(ds['test']['content'])
test_labels = list(ds['test']['label'])

all_texts = train_texts + test_texts
all_labels = train_labels + test_labels

print(f'Total documents: {len(all_texts):,}')
print(f'Train: {len(train_texts):,} | Test: {len(test_texts):,}')

# Show class distribution
class_names = [
    'Company', 'EducationalInstitution', 'Artist', 'Athlete',
    'OfficeHolder', 'MeanOfTransportation', 'Building', 'NaturalPlace',
    'Village', 'Animal', 'Plant', 'Album', 'Film', 'WrittenWork'
]

print(f'\nClass distribution ({len(class_names)} classes):')
unique, counts = np.unique(all_labels, return_counts=True)
for label_idx, count in zip(unique, counts):
    print(f'  {class_names[label_idx]:25s}: {count:,} documents')

In [ ]:
# Subsample to N_DOCS if needed
if len(all_texts) > N_DOCS:
    indices = np.random.choice(len(all_texts), N_DOCS, replace=False)
    texts = [all_texts[i] for i in indices]
    labels = [all_labels[i] for i in indices]
else:
    texts = all_texts
    labels = all_labels

print(f'Using {len(texts):,} documents.')
print(f'\nSample document (truncated):\n{texts[0][:300]}...')
avg_len = np.mean([len(t) for t in texts])
print(f'\nAverage document length: {avg_len:.0f} characters')
print(f'Average word count: {np.mean([len(t.split()) for t in texts]):.0f} words')

## 3. Embedding Generation

In [ ]:
print(f'Loading embedding model: {EMBEDDING_MODEL}...')
model = SentenceTransformer(EMBEDDING_MODEL)

print(f'Generating embeddings for {len(texts):,} documents...')
t0 = time.time()
embeddings = model.encode(
    texts,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True  # L2 normalize for cosine similarity
)
embed_time = time.time() - t0

embeddings = np.array(embeddings, dtype=np.float32)
print(f'\nEmbeddings shape: {embeddings.shape}')
print(f'Embedding time: {embed_time:.1f}s ({len(texts)/embed_time:.0f} docs/sec)')
print(f'Embeddings memory (float32): {embeddings.nbytes / 1e6:.1f} MB')
print(f'Embeddings memory (if 4-bit): {embeddings.nbytes / 1e6 / 8:.1f} MB')
print(f'Vector norms (should be ~1.0): min={np.linalg.norm(embeddings, axis=1).min():.4f}, max={np.linalg.norm(embeddings, axis=1).max():.4f}')

## 4. Semantic Space Partitioning

Each partition becomes an independent memory unit. We use KMeans clustering to create semantically coherent partitions.

In [ ]:
print('Partitioning embedding space into independent semantic memories...\n')

partitions = {}
for n in PARTITION_COUNTS:
    t0 = time.time()
    kmeans = KMeans(n_clusters=n, random_state=RANDOM_SEED, n_init=10, max_iter=300)
    labels = kmeans.fit_predict(embeddings)
    centers = kmeans.cluster_centers_
    # Normalize centers for cosine similarity
    centers = centers / np.linalg.norm(centers, axis=1, keepdims=True)
    
    sil = silhouette_score(embeddings, labels, sample_size=min(10000, len(embeddings)))
    cluster_sizes = [np.sum(labels == i) for i in range(n)]
    
    partitions[n] = {
        'labels': labels,
        'centers': centers,
        'silhouette': sil,
        'sizes': cluster_sizes
    }
    
    elapsed = time.time() - t0
    print(f'K={n:2d} | Silhouette={sil:.4f} | Time={elapsed:.1f}s | Sizes: {cluster_sizes}')

print('\nPartitioning complete.')

# ---------------------------------------------------------------------------
# RANDOM BASELINE: No semantic coherence
# Randomly assign documents to partitions (no clustering)
# ---------------------------------------------------------------------------
print('\n' + '=' * 60)
print('RANDOM BASELINE PARTITIONING (no semantic coherence)')
print('=' * 60)

random_partitions = {}
for n in PARTITION_COUNTS:
    t0 = time.time()
    # Random assignment
    random_labels = np.random.randint(0, n, size=len(embeddings))
    
    # Compute random cluster centers (mean of each random partition)
    random_centers = np.zeros((n, EMBEDDING_DIM))
    for p in range(n):
        mask = random_labels == p
        if mask.sum() > 0:
            random_centers[p] = embeddings[mask].mean(axis=0)
    random_centers = random_centers / np.linalg.norm(random_centers, axis=1, keepdims=True)
    
    sil = silhouette_score(embeddings, random_labels, sample_size=min(10000, len(embeddings)))
    cluster_sizes = [np.sum(random_labels == i) for i in range(n)]
    
    random_partitions[n] = {
        'labels': random_labels,
        'centers': random_centers,
        'silhouette': sil,
        'sizes': cluster_sizes
    }
    
    elapsed = time.time() - t0
    print(f'K={n:2d} | Silhouette={sil:.4f} | Time={elapsed:.1f}s | Sizes: {cluster_sizes}')

print('\nRandom baseline partitioning complete.')

## 5. TurboVec Index Construction per Partition

Each partition gets its own independent TurboVec index. No centralized shared index.

In [ ]:
from turbovec import TurboQuantIndex

def build_partition_indices(embeddings, labels, n_partitions, bit_width=4):
    """Build independent TurboVec indices for each partition."""
    indices = {}
    for p in range(n_partitions):
        mask = labels == p
        partition_vectors = embeddings[mask].astype(np.float32)
        
        idx = TurboQuantIndex(dim=EMBEDDING_DIM, bit_width=bit_width)
        idx.add(partition_vectors)
        
        indices[p] = {
            'index': idx,
            'mask': mask,
            'original_ids': np.where(mask)[0],
            'size': partition_vectors.shape[0]
        }
    return indices

print('Building TurboVec indices for each partition count...\n')
all_indices = {}
for n in PARTITION_COUNTS:
    t0 = time.time()
    idx = build_partition_indices(
        embeddings, partitions[n]['labels'], n, bit_width=BIT_WIDTH
    )
    elapsed = time.time() - t0
    sizes = [idx[p]['size'] for p in range(n)]
    print(f'K={n:2d} | Built {n} indices in {elapsed:.1f}s | Sizes: {sizes}')
    all_indices[n] = idx

# Build global index (baseline)
print('\nBuilding global index (baseline)...')
t0 = time.time()
global_index = TurboQuantIndex(dim=EMBEDDING_DIM, bit_width=BIT_WIDTH)
global_index.add(embeddings.astype(np.float32))
print(f'Global index built in {time.time()-t0:.1f}s ({len(embeddings):,} vectors)')

# Memory comparison
float32_mb = embeddings.nbytes / 1e6
compressed_mb = float32_mb / (32 / BIT_WIDTH)
print(f'\nMemory comparison:')
print(f'  Float32: {float32_mb:.1f} MB')
print(f'  {BIT_WIDTH}-bit compressed: {compressed_mb:.1f} MB ({32/BIT_WIDTH:.0f}x reduction)')
print(f'  Effective per-partition overhead: ~{compressed_mb/len(PARTITION_COUNTS):.1f} MB average')

# Build random baseline indices
print('\nBuilding random baseline indices...')
random_all_indices = {}
for n in PARTITION_COUNTS:
    t0 = time.time()
    idx = build_partition_indices(
        embeddings, random_partitions[n]['labels'], n, bit_width=BIT_WIDTH
    )
    elapsed = time.time() - t0
    sizes = [idx[p]['size'] for p in range(n)]
    print(f'K={n:2d} | Built {n} random indices in {elapsed:.1f}s | Sizes: {sizes}')
    random_all_indices[n] = idx

## 6. Base Search Functions and Confidence Estimation

In [ ]:
def search_global(global_index, query, k=TOP_K):
    """Baseline A: Search the single global index."""
    scores, ids = global_index.search(query.reshape(1, -1).astype(np.float32), k)
    return np.array(scores[0]), np.array(ids[0])


def search_local(indices, query, centers, k=TOP_K):
    """Baseline B: Search only the nearest partition."""
    sims = cosine_similarity(query.reshape(1, -1), centers)[0]
    nearest = np.argmax(sims)
    scores, local_ids = indices[nearest]['index'].search(
        query.reshape(1, -1).astype(np.float32), k
    )
    global_ids = indices[nearest]['original_ids'][np.array(local_ids[0])]
    return np.array(scores[0]), global_ids, nearest


def confidence_estimate(scores, n_contacted, n_total, k=TOP_K):
    """
    Adaptive confidence estimation.
    Penalizes low coverage: few partitions searched = low confidence.
    Formula: sigmoid(gap) * (n_contacted / n_total)
    """
    if len(scores) < 2:
        return 0.0
    n_eval = min(k, len(scores))
    top1 = scores[0]
    rest_mean = np.mean(scores[1:n_eval])
    gap = top1 - rest_mean
    sig = 1.0 / (1.0 + np.exp(-5.0 * gap))
    coverage = n_contacted / n_total
    confidence = sig * coverage
    return float(confidence)


def compute_recall(ground_truth, predicted):
    """Compute Recall@k."""
    gt_set = set(ground_truth)
    hits = sum(1 for p in predicted if p in gt_set)
    return hits / len(ground_truth)


def compute_mrr(ground_truth, predicted):
    """Compute Mean Reciprocal Rank."""
    gt_set = set(ground_truth)
    for i, p in enumerate(predicted):
        if p in gt_set:
            return 1.0 / (i + 1)
    return 0.0


def compute_ndcg(ground_truth, predicted, k=TOP_K):
    """Compute NDCG@k."""
    gt_set = set(ground_truth)
    # Relevance: 1 if in ground truth, 0 otherwise
    relevance = np.array([1.0 if p in gt_set else 0.0 for p in predicted[:k]])
    if relevance.sum() == 0:
        return 0.0
    # DCG
    positions = np.arange(1, len(relevance) + 1)
    dcg = np.sum(relevance / np.log2(positions + 1))
    # Ideal DCG
    ideal_relevance = np.sort(relevance)[::-1]
    idcg = np.sum(ideal_relevance / np.log2(positions + 1))
    return dcg / idcg if idcg > 0 else 0.0


def get_ground_truth(query, embeddings, k=TOP_K):
    """Get ground truth by exact search over all embeddings."""
    sims = cosine_similarity(query.reshape(1, -1), embeddings)[0]
    top_indices = np.argsort(-sims)[:k]
    return set(top_indices)


print('Base search functions defined.')
print('Confidence estimation: sigmoid of (top1_score - mean_rest_scores) * (n_contacted / n_total)')

## 7. Adaptive Routing Strategies

Four communication strategies are implemented:

| Strategy | Communication Policy | Inspiration |
|----------|---------------------|-------------|
| **Confidence Threshold** | Communicate only when local confidence < threshold | Sparse Ising Machines (eta ratio) |
| **Top-N Neighbors** | Always communicate with N closest partitions | HyphaeDB (graph neighbors) |
| **Progressive Expansion** | Expand until confidence target met | HNSW navigation |
| **Budgeted Communication** | Hard limit on partitions contacted | ShardMemo (probe budget) |

In [ ]:
# =============================================================================
# STRATEGY 1: Confidence Threshold
# Only communicate when local retrieval confidence is below threshold.
# Inspired by the eta ratio in Distributed Sparse Ising Machines.
# =============================================================================

def strategy_threshold(indices, query, centers, threshold=0.5, k=TOP_K):
    """
    Search nearest partition first. If confidence < threshold,
    progressively query remaining partitions sorted by distance.
    """
    sims = cosine_similarity(query.reshape(1, -1), centers)[0]
    order = np.argsort(-sims)  # Closest first
    n_total = len(order)
    
    all_scores, all_ids = [], []
    contacted = []
    
    for p in order:
        s, local_ids = indices[p]['index'].search(
            query.reshape(1, -1).astype(np.float32), k
        )
        global_ids = indices[p]['original_ids'][np.array(local_ids[0])]
        all_scores.extend(s[0])
        all_ids.extend(global_ids)
        contacted.append(p)
        
        merged = np.array(all_scores)
        conf = confidence_estimate(merged, len(contacted), n_total, k)
        
        if conf >= threshold:
            break
    
    all_scores = np.array(all_scores)
    all_ids = np.array(all_ids)
    top_idx = np.argsort(-all_scores)[:k]
    return all_scores[top_idx], all_ids[top_idx], contacted, conf


# =============================================================================
# STRATEGY 2: Top-N Neighbor Memories
# Always communicate with the N closest partitions.
# =============================================================================

def strategy_topn(indices, query, centers, n_neighbors=2, k=TOP_K):
    """
    Query the N nearest partitions by centroid distance.
    """
    sims = cosine_similarity(query.reshape(1, -1), centers)[0]
    order = np.argsort(-sims)[:n_neighbors]
    n_total = len(sims)
    
    all_scores, all_ids = [], []
    for p in order:
        s, local_ids = indices[p]['index'].search(
            query.reshape(1, -1).astype(np.float32), k
        )
        global_ids = indices[p]['original_ids'][np.array(local_ids[0])]
        all_scores.extend(s[0])
        all_ids.extend(global_ids)
    
    all_scores = np.array(all_scores)
    all_ids = np.array(all_ids)
    top_idx = np.argsort(-all_scores)[:k]
    conf = confidence_estimate(all_scores[top_idx], n_neighbors, n_total, k)
    return all_scores[top_idx], all_ids[top_idx], list(order), conf


# =============================================================================
# STRATEGY 3: Adaptive Progressive Expansion
# Expand search radius until confidence target is met.
# =============================================================================

def strategy_progressive(indices, query, centers, conf_target=0.5, k=TOP_K):
    """
    Start with nearest partition. Progressively add more
    partitions until confidence reaches target.
    """
    sims = cosine_similarity(query.reshape(1, -1), centers)[0]
    order = np.argsort(-sims)
    n_total = len(order)
    
    all_scores, all_ids = [], []
    contacted = []
    
    for p in order:
        s, local_ids = indices[p]['index'].search(
            query.reshape(1, -1).astype(np.float32), k
        )
        global_ids = indices[p]['original_ids'][np.array(local_ids[0])]
        all_scores.extend(s[0])
        all_ids.extend(global_ids)
        contacted.append(p)
        
        merged = np.array(all_scores)
        conf = confidence_estimate(merged, len(contacted), n_total, k)
        
        if conf >= conf_target:
            break
    
    all_scores = np.array(all_scores)
    all_ids = np.array(all_ids)
    top_idx = np.argsort(-all_scores)[:k]
    return all_scores[top_idx], all_ids[top_idx], contacted, conf


# =============================================================================
# STRATEGY 4: Budgeted Communication
# Hard limit on number of partitions contacted.
# =============================================================================

def strategy_budgeted(indices, query, centers, max_contacted=2, k=TOP_K):
    """
    Query at most max_contacted partitions.
    """
    sims = cosine_similarity(query.reshape(1, -1), centers)[0]
    n_total = len(sims)
    order = np.argsort(-sims)[:max_contacted]
    
    all_scores, all_ids = [], []
    for p in order:
        s, local_ids = indices[p]['index'].search(
            query.reshape(1, -1).astype(np.float32), k
        )
        global_ids = indices[p]['original_ids'][np.array(local_ids[0])]
        all_scores.extend(s[0])
        all_ids.extend(global_ids)
    
    all_scores = np.array(all_scores)
    all_ids = np.array(all_ids)
    top_idx = np.argsort(-all_scores)[:k]
    conf = confidence_estimate(all_scores[top_idx], max_contacted, n_total, k)
    return all_scores[top_idx], all_ids[top_idx], list(order), conf


print('Four adaptive routing strategies defined:')
print('  1. Confidence Threshold (expand when confidence < threshold)')
print('  2. Top-N Neighbors (always query N closest partitions)')
print('  3. Progressive Expansion (expand until confidence target met)')
print('  4. Budgeted Communication (hard limit on partitions contacted)')

## 8. Complete Benchmark

Run all strategies across all partition counts and measure both retrieval quality and communication cost.

In [ ]:
def run_full_benchmark(embeddings, global_index, all_indices, partitions, 
                       n_queries=N_QUERIES, k=TOP_K):
    """
    Benchmark all strategies across all partition counts.
    Returns structured results for analysis.
    """
    n_docs = len(embeddings)
    query_indices = np.random.choice(n_docs, n_queries, replace=False)
    
    strategy_names = ['global', 'local', 'threshold', 'topn', 'progressive', 'budgeted']
    
    results = {n: {s: {'recall': [], 'mrr': [], 'ndcg': [], 'comm': [], 'time': [], 'conf': []}
                   for s in strategy_names}
               for n in PARTITION_COUNTS}
    
    for n_partitions in PARTITION_COUNTS:
        print(f'\nBenchmarking K={n_partitions} partitions...')
        indices = all_indices[n_partitions]
        centers = partitions[n_partitions]['centers']
        
        for qi_idx, qi in enumerate(query_indices):
            query = embeddings[qi]
            gt = get_ground_truth(query, embeddings, k=k)
            
            # --- Baseline A: Global Search ---
            t0 = time.time()
            s_g, i_g = search_global(global_index, query, k)
            t_global = time.time() - t0
            results[n_partitions]['global']['recall'].append(compute_recall(gt, i_g))
            results[n_partitions]['global']['mrr'].append(compute_mrr(gt, i_g))
            results[n_partitions]['global']['ndcg'].append(compute_ndcg(gt, i_g))
            results[n_partitions]['global']['comm'].append(n_partitions)  # contacts all
            results[n_partitions]['global']['time'].append(t_global)
            results[n_partitions]['global']['conf'].append(1.0)
            
            # --- Baseline B: Local Search ---
            t0 = time.time()
            s_l, i_l, nearest = search_local(indices, query, centers, k)
            t_local = time.time() - t0
            results[n_partitions]['local']['recall'].append(compute_recall(gt, i_l))
            results[n_partitions]['local']['mrr'].append(compute_mrr(gt, i_l))
            results[n_partitions]['local']['ndcg'].append(compute_ndcg(gt, i_l))
            results[n_partitions]['local']['comm'].append(1)  # contacts 1
            results[n_partitions]['local']['time'].append(t_local)
            results[n_partitions]['local']['conf'].append(0.0)
            
            # --- Strategy 1: Confidence Threshold ---
            t0 = time.time()
            s_t, i_t, c_t, conf_t = strategy_threshold(indices, query, centers)
            t_t = time.time() - t0
            results[n_partitions]['threshold']['recall'].append(compute_recall(gt, i_t))
            results[n_partitions]['threshold']['mrr'].append(compute_mrr(gt, i_t))
            results[n_partitions]['threshold']['ndcg'].append(compute_ndcg(gt, i_t))
            results[n_partitions]['threshold']['comm'].append(len(c_t))
            results[n_partitions]['threshold']['time'].append(t_t)
            results[n_partitions]['threshold']['conf'].append(conf_t)
            
            # --- Strategy 2: Top-N Neighbors ---
            n_neighbors = max(2, n_partitions // 4)  # Scale with partition count
            t0 = time.time()
            s_n, i_n, c_n, conf_n = strategy_topn(indices, query, centers, n_neighbors)
            t_n = time.time() - t0
            results[n_partitions]['topn']['recall'].append(compute_recall(gt, i_n))
            results[n_partitions]['topn']['mrr'].append(compute_mrr(gt, i_n))
            results[n_partitions]['topn']['ndcg'].append(compute_ndcg(gt, i_n))
            results[n_partitions]['topn']['comm'].append(len(c_n))
            results[n_partitions]['topn']['time'].append(t_n)
            results[n_partitions]['topn']['conf'].append(conf_n)
            
            # --- Strategy 3: Progressive Expansion ---
            t0 = time.time()
            s_p, i_p, c_p, conf_p = strategy_progressive(indices, query, centers)
            t_p = time.time() - t0
            results[n_partitions]['progressive']['recall'].append(compute_recall(gt, i_p))
            results[n_partitions]['progressive']['mrr'].append(compute_mrr(gt, i_p))
            results[n_partitions]['progressive']['ndcg'].append(compute_ndcg(gt, i_p))
            results[n_partitions]['progressive']['comm'].append(len(c_p))
            results[n_partitions]['progressive']['time'].append(t_p)
            results[n_partitions]['progressive']['conf'].append(conf_p)
            
            # --- Strategy 4: Budgeted Communication ---
            max_budget = max(2, n_partitions // 2)  # Budget = half the partitions
            t0 = time.time()
            s_b, i_b, c_b, conf_b = strategy_budgeted(indices, query, centers, max_budget)
            t_b = time.time() - t0
            results[n_partitions]['budgeted']['recall'].append(compute_recall(gt, i_b))
            results[n_partitions]['budgeted']['mrr'].append(compute_mrr(gt, i_b))
            results[n_partitions]['budgeted']['ndcg'].append(compute_ndcg(gt, i_b))
            results[n_partitions]['budgeted']['comm'].append(len(c_b))
            results[n_partitions]['budgeted']['time'].append(t_b)
            results[n_partitions]['budgeted']['conf'].append(conf_b)
        
        # Print summary for this K
        for s in strategy_names:
            avg_recall = np.mean(results[n_partitions][s]['recall'])
            avg_comm = np.mean(results[n_partitions][s]['comm'])
            print(f'  {s:12s} | Recall@{k}={avg_recall:.4f} | Comm={avg_comm:.1f}/{n_partitions}')
    
    return results, query_indices


print('Starting full benchmark...')
print(f'Testing {len(PARTITION_COUNTS)} partition counts x 6 strategies x {N_QUERIES} queries')
t0 = time.time()
results, query_indices = run_full_benchmark(
    embeddings, global_index, all_indices, partitions
)
total_time = time.time() - t0
print(f'\nBenchmark complete in {total_time:.1f}s')

# ---------------------------------------------------------------------------
# RANDOM BASELINE BENCHMARK
# Compare KMeans vs Random partitioning with Threshold strategy
# ---------------------------------------------------------------------------
print('\n' + '=' * 60)
print('RANDOM BASELINE BENCHMARK')
print('=' * 60)

random_results = {n: {'threshold': {'recall': [], 'comm': []}} for n in PARTITION_COUNTS}

for n_partitions in PARTITION_COUNTS:
    print(f'\nBenchmarking K={n_partitions} (random partitions)...')
    indices = random_all_indices[n_partitions]
    centers = random_partitions[n_partitions]['centers']
    
    for qi in query_indices:
        query = embeddings[qi]
        gt = get_ground_truth(query, embeddings, k=TOP_K)
        
        # Threshold strategy on random partitions
        s_t, i_t, c_t, conf_t = strategy_threshold(indices, query, centers)
        random_results[n_partitions]['threshold']['recall'].append(compute_recall(gt, i_t))
        random_results[n_partitions]['threshold']['comm'].append(len(c_t))
    
    avg_recall = np.mean(random_results[n_partitions]['threshold']['recall'])
    avg_comm = np.mean(random_results[n_partitions]['threshold']['comm'])
    kmeans_recall = results[n_partitions]['threshold']['recall'][-1]  # Last query
    print(f'  K={n_partitions:2d} | Random Recall@{TOP_K}={avg_recall:.4f} | KMeans Recall@{TOP_K}={np.mean(results[n_partitions]["threshold"]["recall"]):.4f}')

print('\n' + '=' * 60)
print('KMEANS vs RANDOM COMPARISON (Threshold strategy)')
print('=' * 60)
for n in PARTITION_COUNTS:
    kmeans_r = np.mean(results[n]['threshold']['recall'])
    random_r = np.mean(random_results[n]['threshold']['recall'])
    improvement = kmeans_r - random_r
    print(f'K={n:2d} | KMeans={kmeans_r:.4f} | Random={random_r:.4f} | Improvement={improvement:.4f} ({improvement/random_r*100:.1f}%)')

## 9. Communication Metrics Computation

Communication becomes a first-class metric alongside retrieval quality.

In [ ]:
strategy_names = ['global', 'local', 'threshold', 'topn', 'progressive', 'budgeted']
strategy_labels = ['Global (baseline)', 'Local (baseline)', 'Threshold', 'Top-N', 'Progressive', 'Budgeted']

def compute_all_metrics(results):
    """Compute comprehensive metrics table."""
    metrics_table = []
    
    for n in PARTITION_COUNTS:
        for s_name, s_label in zip(strategy_names, strategy_labels):
            d = results[n][s_name]
            row = {
                'Partitions': n,
                'Strategy': s_label,
                'Recall@10': np.mean(d['recall']),
                'MRR': np.mean(d['mrr']),
                'NDCG@10': np.mean(d['ndcg']),
                'Partitions Contacted': np.mean(d['comm']),
                'Comm Reduction (%)': (1 - np.mean(d['comm']) / n) * 100,
                'Memory Activation (%)': np.mean(d['comm']) / n * 100,
                'Avg Latency (ms)': np.mean(d['time']) * 1000,
                'Avg Confidence': np.mean(d['conf']),
            }
            metrics_table.append(row)
    
    return metrics_table

metrics = compute_all_metrics(results)

# Display as formatted table
import pandas as pd
df = pd.DataFrame(metrics)

print('=' * 120)
print('COMPREHENSIVE RESULTS TABLE')
print('=' * 120)
for n in PARTITION_COUNTS:
    print(f'\n--- {n} Partitions ---')
    subset = df[df['Partitions'] == n].copy()
    subset = subset.round({
        'Recall@10': 4, 'MRR': 4, 'NDCG@10': 4,
        'Partitions Contacted': 2, 'Comm Reduction (%)': 1,
        'Memory Activation (%)': 1, 'Avg Latency (ms)': 2,
        'Avg Confidence': 4
    })
    print(subset.to_string(index=False))

# Summary: best adaptive strategy per partition count
print('\n' + '=' * 80)
print('BEST ADAPTIVE STRATEGY PER PARTITION COUNT')
print('=' * 80)
for n in PARTITION_COUNTS:
    subset = df[(df['Partitions'] == n) & (~df['Strategy'].isin(['Global (baseline)', 'Local (baseline)']))]
    best = subset.loc[subset['Recall@10'].idxmax()]
    global_row = df[(df['Partitions'] == n) & (df['Strategy'] == 'Global (baseline)')].iloc[0]
    recall_loss = global_row['Recall@10'] - best['Recall@10']
    print(f'K={n:2d}: {best["Strategy"]:12s} | Recall={best["Recall@10"]:.4f} (loss={recall_loss:.4f}) | Comm={best["Partitions Contacted"]:.1f}/{n}')

## 10. Visualization

Four publication-ready figures:
1. Recall@10 vs Number of Partitions
2. Communication Cost vs Recall (Pareto frontier)
3. Memory Activation Percentage by Strategy
4. Heatmap: Strategy x Partitions

In [ ]:
colors = {
    'Global (baseline)': '#e74c3c',
    'Local (baseline)': '#95a5a6',
    'Threshold': '#3498db',
    'Top-N': '#2ecc71',
    'Progressive': '#9b59b6',
    'Budgeted': '#f39c12'
}
markers = {
    'Global (baseline)': 's',
    'Local (baseline)': 'D',
    'Threshold': 'o',
    'Top-N': '^',
    'Progressive': 'v',
    'Budgeted': 'p'
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Communication-Efficient Distributed Vector Memory\nDBpedia 14 Dataset (100K documents, 14 semantic classes)', 
             fontsize=14, fontweight='bold', y=1.02)

# ---------------------------------------------------------------------------
# Figure 1: Recall@10 vs Number of Partitions
# ---------------------------------------------------------------------------
ax = axes[0, 0]
for s_label in strategy_labels:
    x_vals = []
    y_vals = []
    for n in PARTITION_COUNTS:
        row = df[(df['Partitions'] == n) & (df['Strategy'] == s_label)]
        if len(row) > 0:
            x_vals.append(n)
            y_vals.append(row['Recall@10'].values[0])
    ax.plot(x_vals, y_vals, marker=markers[s_label], color=colors[s_label],
            label=s_label, linewidth=2, markersize=8)

ax.set_xlabel('Number of Partitions (K)')
ax.set_ylabel('Recall@10')
ax.set_title('Figure 1: Retrieval Quality vs Partition Count')
ax.legend(loc='lower left', fontsize=9)
ax.set_xticks(PARTITION_COUNTS)
ax.grid(True, alpha=0.3)

# ---------------------------------------------------------------------------
# Figure 2: Communication Cost vs Recall (Pareto Frontier)
# ---------------------------------------------------------------------------
ax = axes[0, 1]
for s_label in strategy_labels:
    x_vals = []
    y_vals = []
    for n in PARTITION_COUNTS:
        row = df[(df['Partitions'] == n) & (df['Strategy'] == s_label)]
        if len(row) > 0:
            x_vals.append(row['Partitions Contacted'].values[0])
            y_vals.append(row['Recall@10'].values[0])
    ax.scatter(x_vals, y_vals, marker=markers[s_label], color=colors[s_label],
              label=s_label, s=100, zorder=5)
    ax.plot(x_vals, y_vals, color=colors[s_label], alpha=0.5, linewidth=1)

ax.set_xlabel('Avg Partitions Contacted (Communication Cost)')
ax.set_ylabel('Recall@10')
ax.set_title('Figure 2: Pareto Frontier (Quality vs Communication)')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)

# ---------------------------------------------------------------------------
# Figure 3: Memory Activation Percentage by Strategy
# ---------------------------------------------------------------------------
ax = axes[1, 0]
adaptive_labels = ['Threshold', 'Top-N', 'Progressive', 'Budgeted']
x = np.arange(len(PARTITION_COUNTS))
width = 0.18

for i, s_label in enumerate(adaptive_labels):
    vals = []
    for n in PARTITION_COUNTS:
        row = df[(df['Partitions'] == n) & (df['Strategy'] == s_label)]
        vals.append(row['Memory Activation (%)'].values[0] if len(row) > 0 else 0)
    ax.bar(x + i * width, vals, width, label=s_label, color=colors[s_label], alpha=0.85)

ax.set_xlabel('Number of Partitions (K)')
ax.set_ylabel('Memory Activation (%)')
ax.set_title('Figure 3: Percentage of Memories Activated per Query')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([str(n) for n in PARTITION_COUNTS])
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# ---------------------------------------------------------------------------
# Figure 4: Heatmap - Strategy x Partitions
# ---------------------------------------------------------------------------
ax = axes[1, 1]
heatmap_data = np.zeros((len(strategy_labels), len(PARTITION_COUNTS)))
for i, s_label in enumerate(strategy_labels):
    for j, n in enumerate(PARTITION_COUNTS):
        row = df[(df['Partitions'] == n) & (df['Strategy'] == s_label)]
        if len(row) > 0:
            heatmap_data[i, j] = row['Recall@10'].values[0]

im = ax.imshow(heatmap_data, cmap='RdYlGn', aspect='auto', vmin=0.3, vmax=1.0)
ax.set_xticks(range(len(PARTITION_COUNTS)))
ax.set_xticklabels([str(n) for n in PARTITION_COUNTS])
ax.set_yticks(range(len(strategy_labels)))
ax.set_yticklabels(strategy_labels)
ax.set_xlabel('Number of Partitions (K)')
ax.set_title('Figure 4: Recall@10 Heatmap')

# Add text annotations
for i in range(len(strategy_labels)):
    for j in range(len(PARTITION_COUNTS)):
        text = ax.text(j, i, f'{heatmap_data[i, j]:.3f}',
                       ha='center', va='center', color='black', fontsize=9, fontweight='bold')

plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.savefig('dbpedia14_distributed_memory_results.png', dpi=150, bbox_inches='tight')
plt.show()

print('Figures saved to dbpedia14_distributed_memory_results.png')

## Summary and Key Findings

In [ ]:
print('=' * 80)
print('KEY FINDINGS SUMMARY')
print('=' * 80)

for n in PARTITION_COUNTS:
    global_recall = df[(df['Partitions'] == n) & (df['Strategy'] == 'Global (baseline)')]['Recall@10'].values[0]
    local_recall = df[(df['Partitions'] == n) & (df['Strategy'] == 'Local (baseline)')]['Recall@10'].values[0]
    
    print(f'\nK={n} Partitions:')
    print(f'  Q1: Quality loss with single partition: {(global_recall - local_recall):.4f} ({(global_recall - local_recall)/global_recall*100:.1f}%)')
    
    # Best adaptive strategy
    best_row = None
    best_f1 = -1
    for s_label in ['Threshold', 'Top-N', 'Progressive', 'Budgeted']:
        row = df[(df['Partitions'] == n) & (df['Strategy'] == s_label)]
        if len(row) > 0:
            # F1-like score: balance recall and communication efficiency
            recall = row['Recall@10'].values[0]
            comm_ratio = 1 - row['Partitions Contacted'].values[0] / n
            f1 = 2 * recall * comm_ratio / (recall + comm_ratio + 1e-8)
            if f1 > best_f1:
                best_f1 = f1
                best_row = row.iloc[0]
                best_name = s_label
    
    if best_row is not None:
        recall_gap = global_recall - best_row['Recall@10']
        comm_saved = best_row['Comm Reduction (%)']
        print(f'  Q2: Best adaptive = {best_name}: Recall={best_row["Recall@10"]:.4f} (gap={recall_gap:.4f}), Comm saved={comm_saved:.1f}%')
        print(f'  Q3: Avg partitions contacted: {best_row["Partitions Contacted"]:.1f} out of {n}')

print(f'\n{"=" * 80}')
print('Q4: Does communication grow linearly with memory size?')
print('    See Figure 2 (Pareto frontier) for the communication-quality tradeoff.')
print(f'\nQ5: Can routing outperform broadcasting?')
threshold_data = df[df['Strategy'] == 'Threshold']
for _, row in threshold_data.iterrows():
    k = int(row['Partitions'])
    global_row = df[(df['Partitions'] == k) & (df['Strategy'] == 'Global (baseline)')]
    if len(global_row) > 0:
        g_recall = global_row['Recall@10'].values[0]
        ratio = row['Partitions Contacted'] / k
        if row['Recall@10'] >= 0.95 * g_recall:
            print(f'    K={k}: YES - Threshold achieves {row["Recall@10"]:.4f} recall (>=95% of global {g_recall:.4f}) with only {ratio*100:.0f}% communication')
        else:
            print(f'    K={k}: Partial - Threshold achieves {row["Recall@10"]:.4f} recall ({row["Recall@10"]/g_recall*100:.1f}% of global) with {ratio*100:.0f}% communication')

print(f'\n{"=" * 80}')
print('Q6: Does semantic partitioning matter? (KMeans vs Random)')
print('    See Random Baseline Benchmark results above.')

print(f'\n{"=" * 80}')
print('EXPERIMENT COMPLETE')
print(f'Corpus: {len(texts):,} documents from DBpedia 14')
print(f'Engine: TurboVec (TurboQuant, ICLR 2026) at {BIT_WIDTH}-bit')
print(f'Partition counts: {PARTITION_COUNTS}')
print(f'Partitioning: KMeans vs Random baseline')
print(f'Queries evaluated: {N_QUERIES}')
print(f'Total benchmark time: {total_time:.1f}s')